# Audit of the fitting notebook

This notebook checks whether the models in `fitting.ipynb` are safe to put in the report. It reruns everything it needs, so you can read it on its own.

There are six checks. Each one ends with a verdict in bold.

1. What the two formulas are.
2. Lookahead: do the inputs already know the answer?
3. Multicollinearity: are the three kernel numbers copies of each other?
4. Sample size: are there enough grokked runs?
5. Weight decay: is it a constant effect, and is it fair to swap levels?
6. Is a big L1 model worth doing?

The short version is at the top. The details follow.

## The short version

| Check | Verdict | What it means for the paper |
| --- | --- | --- |
| 1. Formulas | Fine. | Write them as in Section 1. |
| 2. Lookahead | Clean. | The kernel on the training pairs alone scores the same as the whole probe. |
| 3. Multicollinearity | `A_t` and `S_t` are near twins across runs (r = 0.91, VIF up to 42). Within cells they hardly overlap. | Report the three numbers as a block (joint test). Don't rank them by their AFT or logistic coefficients. Drop the Part 8 claim that `R_t` matters most. |
| 4. Sample size | Pooled set: fine. Weight decay 3e-5: fine for "when", too few failures for "whether". No weight decay: thin. | Lead with the pooled set. Use the no weight decay set as a side check. |
| 5. Weight decay | Not a constant effect. It shifts the time, changes the spread, and changes how the kernel numbers line up with the time across cells. The within-cell effect of `A_t` is the same at every level. | Give each level its own term. Don't train on one level and test on another. Lead with the within-cell result. |
| 6. Big L1 | Not worth a section. It scores higher, but mostly by rebuilding weight decay, and it ignores `A_t`, `S_t` and `R_t`. | One sentence in the appendix at most. |

## Setup

Same data and same definitions as Parts 5 to 7 of `fitting.ipynb`. The grokking step `T` is the first checkpoint with test accuracy of at least 0.95. Runs that never get there are censored at step 200,000. The kernel numbers are read at step 2,000. Weight decay 3e-3 is left out because none of its runs memorise.

In [1]:
import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import chi2
from statsmodels.stats.outliers_influence import variance_inflation_factor
from lifelines import CoxPHFitter, LogNormalAFTFitter
from lifelines.utils import concordance_index
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut

warnings.filterwarnings('ignore')
LANDMARK, WD, BUDGET = 2_000, 3e-5, 200_000
inputs = ['A_t', 'S_c', 'R_c']
names = {'A_t': 'A_t', 'S_c': 'S_t', 'R_c': 'R_t'}
SETTINGS = ['alpha', 'width', 'eta_kappa']

runs = pd.read_parquet('data/runs.parquet')
acc = pd.read_parquet('data/checkpoints.parquet', columns=['run_id', 'step', 'train_acc', 'test_acc'])
events = runs[['run_id', 'alpha', 'width', 'eta_kappa', 'seed', 'last_step']].set_index('run_id')
events['t_gen95'] = acc[acc['test_acc'] >= 0.95].groupby('run_id')['step'].min()
events = events.reset_index()
events['grokked'] = events['t_gen95'].notna().astype(int)
events['T'] = events['t_gen95'].fillna(events['last_step'])
events['cell'] = (events['alpha'].astype(str) + '_' + events['width'].astype(str) + '_'
                  + events['eta_kappa'].map('{:g}'.format))

at_landmark = pd.read_parquet('data/checkpoints.parquet', filters=[('step', '==', LANDMARK)])
data = events.merge(at_landmark.drop(columns=['alpha', 'width', 'eta_kappa', 'seed']), on='run_id')
data = data[data['eta_kappa'] <= 1e-3].reset_index(drop=True)

POOLED = 'all levels up to 1e-3'
sets = {'weight decay 3e-5': data[np.isclose(data['eta_kappa'], WD)].reset_index(drop=True),
        'no weight decay': data[data['eta_kappa'] == 0].reset_index(drop=True),
        POOLED: data}
pooled = sets[POOLED]

In [2]:
def design(df, cols, settings, ref):
    """As in Part 7: one indicator per setting value, and kernel numbers scaled with the mean and spread of `ref`."""
    X = pd.DataFrame(index=df.index)
    for setting in settings:
        for value in sorted(ref[setting].unique())[1:]:
            X[f'{setting}={value:g}'] = (df[setting] == value).astype(float)
    for col in cols:
        X[names.get(col, col)] = (df[col] - ref[col].mean()) / ref[col].std()
    return X

def fit_aft(df, cols, settings, ref=None, extra=None, ancillary=None):
    ref = df if ref is None else ref
    X = design(df, cols, settings, ref)
    if extra is not None:
        X = X.join(extra)
    table = X.assign(T=df['T'], grokked=df['grokked'])
    return LogNormalAFTFitter(penalizer=0.01).fit(table, 'T', 'grokked', ancillary=ancillary)

def held_out_medians(df, cols, settings):
    """Leave one cell out, as in Part 7."""
    pred = pd.Series(index=df.index, dtype=float)
    for train, test in LeaveOneGroupOut().split(df, groups=df['cell']):
        tr, te = df.iloc[train], df.iloc[test]
        pred.iloc[test] = fit_aft(tr, cols, settings).predict_median(design(te, cols, settings, tr)).to_numpy().ravel()
    return pred

def c_index(df, pred):
    return concordance_index(df['T'], pred, df['grokked'])

def held_out_auc(df, cols, settings):
    """Leave one cell out, as in Part 8."""
    prob = pd.Series(index=df.index, dtype=float)
    for train, test in LeaveOneGroupOut().split(df, groups=df['cell']):
        tr, te = df.iloc[train], df.iloc[test]
        model = LogisticRegression(max_iter=1000).fit(design(tr, cols, settings, tr), tr['grokked'])
        prob.iloc[test] = model.predict_proba(design(te, cols, settings, tr))[:, 1]
    return roc_auc_score(df['grokked'], prob)

## 1. The two formulas

**The "linear regression" (Part 7).** This is a log-normal accelerated failure time (AFT) model. Think of it as a linear regression on the log of the grokking step that also knows what to do with runs that never grok.

$$\log T_i = \beta_0 + \sum_j \beta_j \, \mathbb{1}[\text{run } i \text{ has setting value } j] + \beta_A A_i + \beta_S S_i + \beta_R R_i + \sigma \varepsilon_i, \qquad \varepsilon_i \sim N(0, 1)$$

- $T_i$ is the first step at which run $i$ has test accuracy of at least 0.95.
- The sum runs over the values of $\alpha$, width and weight decay. Each value gets its own indicator, so the model doesn't assume that width or $\alpha$ act in a straight line.
- $A_i$, $S_i$ and $R_i$ are the kernel numbers at step 2,000, scaled to mean 0 and standard deviation 1.
- A run that hasn't grokked by step 200,000 adds $P(T_i > 200{,}000)$ to the likelihood. Ordinary least squares would have to drop it, and that would drop exactly the slowest runs.
- How to read a coefficient: $\beta_A = -0.3$ means one standard deviation more of $A_t$ goes with a grokking step $e^{-0.3} \approx 0.74$ times as long.

**The logistic regression (Part 8).**

$$\operatorname{logit} P(\text{run } i \text{ groks by step } 200{,}000) = \gamma_0 + \sum_j \gamma_j \, \mathbb{1}[\text{run } i \text{ has setting value } j] + \gamma_A A_i + \gamma_S S_i + \gamma_R R_i$$

Same inputs. This one asks whether a run groks, and the AFT asks when.

Two small quirks to know about:

- scikit-learn's `LogisticRegression` adds a ridge penalty by default (`C = 1`). That's fine for scores like AUC. The coefficients are shrunk a little, so don't quote them as plain logistic coefficients.
- The AFT has a ridge penalty of 0.01. That's small enough to ignore.

**Verdict: both formulas suit this data. Write them like this in the report.**

## 2. Lookahead

Lookahead means an input carries information from after the moment we claim to predict from. Here that moment is step 2,000. Each line below is one way it could sneak in.

In [3]:
pd.Series({
    'every input is read at step 2,000': (data['step'] == LANDMARK).all(),
    'no run reaches test accuracy 0.95 by step 2,000': events['T'].min() > LANDMARK,
    'no run has test accuracy above 0.2 at step 2,000': acc.loc[acc['step'] == LANDMARK, 'test_acc'].max() < 0.2,
    'every run has the same budget, so being censored says nothing extra': (runs['last_step'] == BUDGET).all(),
}, name='passes')

every input is read at step 2,000                                      True
no run reaches test accuracy 0.95 by step 2,000                        True
no run has test accuracy above 0.2 at step 2,000                       True
every run has the same budget, so being censored says nothing extra    True
Name: passes, dtype: bool

Three more pass when you read the code:

- The kernel numbers are scaled with the mean and spread of the training fold only (`design(..., ref=tr)`). The held-out cell never sets the scale.
- No column measured at or after grokking goes into any model. Part 6 of `fitting.ipynb` lists those columns (`*_at_gen`, `*_final`, `A_peak`, `t_A*`, `weight_norm_growth`).
- The folds hold out whole cells. A run's near-twin seeds are never in training while it is being scored.

**One possible leak to test.** The probe has 256 pairs, and 53 of them come from the test set. So $A_t$ uses the labels of 53 test pairs. That's information from the test set, even though it's read early. If the signal came from those 53 labels, a kernel built on the training pairs alone should predict worse. Here are the held-out concordances of the settings + kernel model both ways.

In [4]:
probe_parts = {'whole probe': inputs, 'training pairs only': ['A_t_train', 'S_c_train', 'R_c_train']}
loco_pred = {(label, part): held_out_medians(df, cols, SETTINGS)
             for label, df in sets.items() for part, cols in probe_parts.items()}
pd.DataFrame({label: {part: c_index(df, loco_pred[(label, part)]) for part in probe_parts}
              for label, df in sets.items()}).round(3)

,weight decay 3e-5,no weight decay,all levels up to 1e-3
whole probe,0.940,0.968,0.901
training pairs only,0.946,0.964,0.901


**Verdict: no lookahead.** The kernel on the training pairs alone scores the same as the whole probe: 0.946 against 0.940 at weight decay 3e-5, 0.964 against 0.968 without weight decay, and 0.901 for both pooled. So the 53 test labels aren't doing the work.

One softer issue. Weight decay 3e-5 was picked as the main level after we saw that most of its runs grok. That's a choice made by looking at the outcomes. The pooled set uses every level up to 1e-3, so lead with the pooled set in the paper.

## 3. Multicollinearity

If two inputs move together, the model can't tell which one deserves the credit. The fit still predicts fine, but the single coefficients get shaky and can even flip sign.

Two things to look at:

- The correlation between each pair of numbers. "Across runs" uses every run. "Within cells" first subtracts each cell's mean, so it only sees how seeds of the same cell differ. The within-cell Cox model of Part 7 uses only that second kind of variation.
- The variance inflation factor (VIF). A VIF of 1 means no overlap. Above about 10 is the usual warning line.

In [5]:
def correlations(df):
    within = df[inputs] - df.groupby('cell')[inputs].transform('mean')
    out = {}
    for kind, frame in {'across runs': df[inputs], 'within cells': within}.items():
        c = frame.corr()
        for a, b in [('A_t', 'S_c'), ('A_t', 'R_c'), ('S_c', 'R_c')]:
            out[(kind, f'{names[a]} and {names[b]}')] = c.loc[a, b]
    return pd.Series(out)

pd.DataFrame({label: correlations(df) for label, df in sets.items()}).round(2)

weight decay 3e-5  no weight decay  \
across runs  A_t and S_t               0.91             0.92   
             A_t and R_t               0.56             0.54   
             S_t and R_t               0.77             0.75   
within cells A_t and S_t               0.30             0.21   
             A_t and R_t               0.20             0.18   
             S_t and R_t              -0.08            -0.05   

                          all levels up to 1e-3  
across runs  A_t and S_t                   0.75  
             A_t and R_t                   0.36  
             S_t and R_t                   0.30  
within cells A_t and S_t                   0.09  
             A_t and R_t                   0.19  
             S_t and R_t                  -0.16

In [6]:
def vifs(df, settings):
    X = sm.add_constant(design(df, inputs, settings, df))
    return {names[c]: variance_inflation_factor(X.to_numpy(), X.columns.get_loc(names[c])) for c in inputs}

pd.DataFrame({(label, 'with the settings' if s else 'kernel only'): vifs(df, s)
              for label, df in sets.items() for s in [[], SETTINGS]}).T.round(1)

A_t   S_t   R_t
weight decay 3e-5     kernel only         8.7  14.7   3.6
                      with the settings  30.5  42.0  11.8
no weight decay       kernel only         9.6  15.2   3.3
                      with the settings  28.0  36.9  12.2
all levels up to 1e-3 kernel only         2.4   2.3   1.2
                      with the settings   7.6  12.9   5.9

**Does this change the drop-one table of Part 8?** Part 8 found that dropping `R_t` from the pooled logistic model cost the most. But when two inputs overlap, dropping one of them costs little because the other covers for it. So we also drop `A_t` and `S_t` together.

In [7]:
drops = {'keep all three': inputs, 'drop R_t': ['A_t', 'S_c'], 'drop A_t': ['S_c', 'R_c'],
         'drop S_t': ['A_t', 'R_c'], 'drop A_t and S_t together': ['R_c'], 'drop all three': []}
pd.Series({name: held_out_auc(pooled, cols, SETTINGS) for name, cols in drops.items()},
          name='held-out AUC, pooled').round(3)

keep all three               0.965
drop R_t                     0.958
drop A_t                     0.960
drop S_t                     0.964
drop A_t and S_t together    0.964
drop all three               0.950
Name: held-out AUC, pooled, dtype: float64

**Verdict: lots of overlap across runs, very little within cells.**

- Inside one weight decay level, `A_t` and `S_t` are near twins across runs (r = 0.91 and 0.92). With the settings in the model their VIFs are 28 to 42, far over the warning line. Pooled, it's milder (VIF 6 to 13).
- That explains the odd signs in Part 7, such as the positive `S_t` in the pooled AFT. Section 5a shows that `S_t` even flips sign from level to level. So don't read the AFT or logistic coefficients of `A_t`, `S_t` and `R_t` one at a time. Report the joint test of the three together.
- Within cells the overlap is small (r between -0.16 and 0.30). So the within-cell Cox coefficient of `A_t` in Part 7 is safe to read on its own. That's the one to quote.
- The drop-one table doesn't work as a ranking. Each single drop moves the AUC by 0.007 or less. Dropping `A_t` and `S_t` together costs less than dropping `A_t` alone, which tells you these gaps are noise. So the Part 8 claim that `R_t` matters most should go. What holds is that the three numbers together add about 0.015 AUC on top of the settings (0.950 to 0.965).

## 4. Are there enough grokked runs?

A common rule of thumb asks for at least 10 events per coefficient. For the AFT the events are the grokked runs. For the logistic model it's whichever class is smaller.

There's a catch. The 5 seeds of a cell are near copies: Part 6 found that 98 percent of the spread in the grokking step sits between cells. So the honest count is closer to the number of cells than the number of runs. The table shows both.

In [8]:
rows = {}
for label, df in sets.items():
    k = design(df, inputs, SETTINGS, df).shape[1]
    grok_cells = df.loc[df['grokked'] == 1, 'cell'].nunique()
    rows[label] = {'coefficients': k, 'cells': df['cell'].nunique(),
                   'grokked runs': df['grokked'].sum(), 'runs not grokked': (1 - df['grokked']).sum(),
                   'cells with a grokked run': grok_cells,
                   'cells with a run not grokked': df.loc[df['grokked'] == 0, 'cell'].nunique(),
                   'grokked runs per coefficient': df['grokked'].sum() / k,
                   'cells with a grokked run per coefficient': grok_cells / k}
pd.DataFrame(rows).round(1)

,weight decay 3e-5,no weight decay,all levels up to 1e-3
coefficients,10.0,10.0,16.0
cells,18.0,18.0,126.0
grokked runs,86.0,23.0,355.0
runs not grokked,4.0,67.0,275.0
cells with a grokked run,18.0,5.0,75.0
cells with a run not grokked,1.0,14.0,57.0
grokked runs per coefficient,8.6,2.3,22.2
cells with a grokked run per coefficient,1.8,0.5,4.7


**Verdict: the pooled set is fine. The single levels are thin.**

- **Pooled.** 355 grokked runs for 16 coefficients, about 22 per coefficient. Counting cells instead, 75 cells with a grokked run, about 5 per coefficient. That's enough for the joint tests and the held-out scores.
- **Weight decay 3e-5.** 86 grokked runs, but only 18 cells for 10 coefficients. It works for "when", and the held-out concordance guards against overfitting. It can't do "whether": the 4 runs that don't grok all sit in one cell. That's why Part 8 skips this set.
- **No weight decay.** 23 grokked runs in 5 cells for 10 coefficients. That's too thin to say anything about single coefficients. Use it as a side check only. Its high concordance (0.97) comes from the settings alone.

## 5. Weight decay: a constant effect, and is it fair to swap levels?

Three questions.

- **5a.** Does each kernel number have the same effect at every weight decay level?
- **5b.** Does weight decay only slide the grokking time along, or does it also change the spread?
- **5c.** If we train on some levels, can we predict a level the model has never seen?

### 5a. Same kernel effect at every level?

We add an interaction for each kernel number with each weight decay level to the pooled AFT, and test whether those 18 extra coefficients help. The test treats the seeds as independent runs, which makes it too confident. So we also show the p-value after cutting the evidence by a factor of 5, which is roughly what you'd get if each cell counted as one run.

In [9]:
base_X = design(pooled, inputs, SETTINGS, pooled)
wd_cols = [c for c in base_X if c.startswith('eta_kappa=')]
interactions = pd.DataFrame({f'{k} x {w}': base_X[k] * base_X[w] for k in names.values() for w in wd_cols})
base = fit_aft(pooled, inputs, SETTINGS)
with_inter = fit_aft(pooled, inputs, SETTINGS, extra=interactions)
lr = 2 * (with_inter.log_likelihood_ - base.log_likelihood_)
pd.Series({'extra coefficients': interactions.shape[1], 'likelihood ratio': round(lr, 1),
           'p': f'{chi2.sf(lr, interactions.shape[1]):.1g}',
           'p with the evidence cut by 5': f'{chi2.sf(lr / 5, interactions.shape[1]):.1g}'})

extra coefficients                 18
likelihood ratio                279.0
p                               1e-48
p with the evidence cut by 5    1e-05
dtype: object

Here are the kernel coefficients when the AFT is fitted on each level by itself, with indicators for `alpha` and width.

In [10]:
per_level = {}
for level, df in pooled.groupby('eta_kappa'):
    s = fit_aft(df.reset_index(drop=True), inputs, SETTINGS).summary.loc['mu_'].loc[list(names.values())]
    per_level[(f'{level:g}', df['grokked'].sum())] = s['coef'].round(2).astype(str) + np.where(s['p'] < 0.05, ' *', '')
per_level = pd.DataFrame(per_level).T
per_level.index.names = ['weight decay', 'grokked runs']
per_level

,covariate,A_t,S_t,R_t
weight decay,grokked runs,,,
0,23,-0.3,-0.87,-0.06
3e-06,40,-0.28,-0.18,0.41 *
1e-05,64,-0.35 *,-0.43 *,0.26 *
3e-05,86,-0.33 *,0.13,-0.28 *
0.0001,77,-0.03,0.82 *,-0.56 *
0.0003,50,0.1,2.03 *,-0.41 *
0.001,15,-0.23,-1.17 *,0.24


A star marks p < 0.05. The same question for the within-cell Cox model, which is the main result of Part 7: does the effect of `A_t` among the seeds of one cell change with the weight decay level?

In [11]:
cox_table = pooled[['T', 'grokked', 'cell']].join(design(pooled, inputs, [], pooled))
wd_dummies = pd.get_dummies(pooled['eta_kappa'].map('{:g}'.format), prefix='wd', drop_first=True).astype(float)
cox_inter = cox_table.join(pd.DataFrame({f'A_t x {w}': cox_table['A_t'] * wd_dummies[w] for w in wd_dummies}))
cox_base = CoxPHFitter().fit(cox_table, 'T', 'grokked', strata='cell')
cox_full = CoxPHFitter().fit(cox_inter, 'T', 'grokked', strata='cell')
lr = 2 * (cox_full.log_likelihood_ - cox_base.log_likelihood_)
pd.Series({'extra coefficients': wd_dummies.shape[1], 'likelihood ratio': round(lr, 1),
           'p': round(chi2.sf(lr, wd_dummies.shape[1]), 3)})

extra coefficients    6.000
likelihood ratio      1.800
p                     0.935
dtype: float64

### 5b. Does weight decay only slide the time along?

The AFT above gives each weight decay level its own shift in the log grokking step, but the same spread $\sigma$ for all of them. We let $\sigma$ differ by level and test whether that helps.

In [12]:
spread = fit_aft(pooled, inputs, SETTINGS, ancillary=base_X[wd_cols])
lr = 2 * (spread.log_likelihood_ - base.log_likelihood_)
sig = spread.summary.loc['sigma_', 'coef']
sigma_by_level = pd.Series({'0': np.exp(sig['Intercept']),
                            **{c.split('=')[1]: np.exp(sig['Intercept'] + sig[c]) for c in wd_cols}})
print(f'likelihood ratio {lr:.1f} on {len(wd_cols)} extra coefficients, p = {chi2.sf(lr, len(wd_cols)):.1g}')
sigma_by_level.rename('sigma, spread of log T').round(2)

likelihood ratio 169.2 on 6 extra coefficients, p = 7e-34


0         1.66
3e-06     1.11
1e-05     0.50
3e-05     0.19
0.0001    0.50
0.0003    1.27
0.001     1.95
Name: sigma, spread of log T, dtype: float64

### 5c. Can the model predict a weight decay level it has never seen?

We hold out one whole weight decay level, train on the other six, and predict the held-out level. A model without weight decay indicators has to work out the level's effect from what it sees. We score each model in two ways.

- **Across all runs.** This asks whether the model puts the held-out levels in the right place relative to each other.
- **Within a level, averaged.** This asks whether the model ranks the runs inside a level correctly.

The last two rows are the leave-one-cell-out scores, where every level is in training.

In [13]:
def leave_one_level_out(df, cols, settings):
    pred = pd.Series(index=df.index, dtype=float)
    for level in df['eta_kappa'].unique():
        te, tr = df[df['eta_kappa'] == level], df[df['eta_kappa'] != level]
        pred[te.index] = fit_aft(tr, cols, settings).predict_median(design(te, cols, settings, tr)).to_numpy().ravel()
    return pred

def two_scores(pred):
    within = [c_index(df, pred[df.index]) for _, df in pooled.groupby('eta_kappa')]
    return {'across all runs': c_index(pooled, pred), 'within a level, averaged': np.mean(within)}

swap_models = {'alpha + width': ([], ['alpha', 'width']), 'kernel only': (inputs, []),
               'alpha + width + kernel': (inputs, ['alpha', 'width'])}
swap = {f'{name}, level held out': two_scores(leave_one_level_out(pooled, cols, s))
        for name, (cols, s) in swap_models.items()}
swap['all settings, cell held out'] = two_scores(held_out_medians(pooled, [], SETTINGS))
swap['all settings + kernel, cell held out'] = two_scores(loco_pred[(POOLED, 'whole probe')])
pd.DataFrame(swap).T.round(3)

,across all runs,"within a level, averaged"
"alpha + width, level held out",0.765,0.923
"kernel only, level held out",0.766,0.900
"alpha + width + kernel, level held out",0.729,0.919
"all settings, cell held out",0.877,0.918
"all settings + kernel, cell held out",0.901,0.924


**Verdict: weight decay isn't a constant effect. Keep it as its own term and don't swap levels.**

- **5a, across cells.** The kernel numbers don't have the same effect at every level (p = 1e-5 even with the evidence cut by 5). `A_t` sits at about -0.3 from no weight decay up to 3e-5, then fades to near 0 from 1e-4 upward. `S_t` and `R_t` change sign between levels. So the pooled AFT coefficients are an average over levels that disagree.
- **5a, within cells.** Here the answer flips. The effect of `A_t` among the seeds of one cell is the same at every level (p = 0.94). This is the cleanest finding in the whole notebook: a seed with higher alignment at step 2,000 groks sooner than its siblings, whatever the weight decay.
- **5b.** Weight decay changes the spread as well as the typical time (p = 7e-34). The spread of log T is smallest at 3e-5 (0.19) and grows on both sides, to 1.66 without weight decay and 1.95 at 1e-3. That's why the Kaplan-Meier curves in Part 5 cross.
- **5c.** A model that hasn't seen a level can still rank the runs inside it (0.90 to 0.92 within a level). It can't place that level against the others: across all runs the concordance drops from 0.88 to 0.90 down to 0.73 to 0.77. The kernel at step 2,000 doesn't carry the effect of weight decay either. Adding it to `alpha` and width makes the cross-level score worse (0.729).

**So, is swapping levels fair?** Training on 3e-5 and testing on no weight decay mixes two things: whether the kernel predicts grokking, and how much the level itself moves the answer. The fair designs are the ones already in `fitting.ipynb`. Either fit one level at a time, or pool every level with one indicator per level and hold out cells. In the paper, say that weight decay sets its own baseline time and spread, and that the model isn't meant to predict a level it hasn't seen.

## 6. Is a big L1 model worth it?

The idea is to put every number logged at step 2,000 into a logistic model with an L1 penalty, and let the penalty pick the ones that matter. There are about 100 of them. Anything measured on the test set is left out (`test_*`, `*_test`, `krr_*`, `A_full`), so nothing peeks.

The penalty strength is picked inside each training fold. All models in the table use the same 10 folds, and each fold holds out whole cells.

In [14]:
skip = {'run_id', 'step', 'log10_step1', 'dt_prev', 'on_linear_grid', 'on_log_grid', 'wall_seconds',
        'stationary', 'kernel_degenerate', 'A_full'}
l1_features = [c for c in at_landmark.columns
               if c not in skip | {'alpha', 'width', 'eta_kappa', 'seed'}
               and not c.startswith(('test_', 'krr_')) and not c.endswith('_test')
               and pooled[c].notna().all() and pooled[c].std() > 0]
Cs = [0.01, 0.03, 0.1, 0.3, 1, 3, 10]

def l1(C):
    return LogisticRegression(l1_ratio=1, C=C, solver='liblinear', max_iter=5000, random_state=0)

def pick_C(X, y, groups):
    inner = list(GroupKFold(5).split(X, y, groups))
    return max(Cs, key=lambda C: np.mean([roc_auc_score(y.iloc[b], l1(C).fit(X.iloc[a], y.iloc[a]).predict_proba(X.iloc[b])[:, 1])
                                          for a, b in inner]))

folds = list(GroupKFold(10).split(pooled, groups=pooled['cell']))

def grouped_auc(cols, settings, use_l1=False):
    prob = np.zeros(len(pooled))
    for train, test in folds:
        tr, te = pooled.iloc[train], pooled.iloc[test]
        X_tr, y_tr = design(tr, cols, settings, tr), tr['grokked']
        model = l1(pick_C(X_tr, y_tr, tr['cell'])) if use_l1 else LogisticRegression(max_iter=1000)
        prob[test] = model.fit(X_tr, y_tr).predict_proba(design(te, cols, settings, tr))[:, 1]
    return roc_auc_score(pooled['grokked'], prob)

print(len(l1_features), 'features go into the L1 model')
pd.Series({'settings only': grouped_auc([], SETTINGS),
           'the 3 kernel numbers only': grouped_auc(inputs, []),
           'settings + the 3 kernel numbers': grouped_auc(inputs, SETTINGS),
           f'L1 on all {len(l1_features)} numbers': grouped_auc(l1_features, [], use_l1=True)},
          name='held-out AUC, pooled').round(3)

108 features go into the L1 model


settings only                      0.948
the 3 kernel numbers only          0.886
settings + the 3 kernel numbers    0.960
L1 on all 108 numbers              0.983
Name: held-out AUC, pooled, dtype: float64

Which numbers does it pick, and does it pick the same ones every time? We redraw the cells with replacement 100 times, refit, and count how often each number survives the penalty. The last column is the share of each number's spread that the weight decay level alone explains.

The table also shows some near copies of the report numbers that are in the list: `A_t_train` and `A_first_logit` for `A_t` (r = 0.99), `S_full` and `S_W1` for `S_t` (r = 1.00), and `share_k0` and `R_full` for `R_t` (r = 1.00).

In [15]:
X_all, y_all = design(pooled, l1_features, [], pooled), pooled['grokked']
best_C = pick_C(X_all, y_all, pooled['cell'])
rng = np.random.default_rng(0)
cell_rows = pooled.groupby('cell').indices
picked = pd.Series(0.0, index=X_all.columns)
for _ in range(100):
    rows = np.concatenate([cell_rows[c] for c in rng.choice(list(cell_rows), len(cell_rows))])
    picked += l1(best_C).fit(X_all.iloc[rows], y_all.iloc[rows]).coef_.ravel() != 0
picked /= 100
wd_share = X_all.groupby(pooled['eta_kappa']).transform('mean').var() / X_all.var()
l1_table = pd.DataFrame({'picked in this share of redraws': picked, 'share of spread set by weight decay': wd_share})
l1_table = l1_table.sort_values('picked in this share of redraws', ascending=False)
l1_table['rank'] = np.arange(1, len(l1_table) + 1)
print(f'penalty C = {best_C}; numbers picked in at least half the redraws: {(picked >= 0.5).sum()}')
copies = ['A_t_train', 'A_first_logit', 'S_full', 'S_W1', 'share_k0', 'R_full']
pd.concat({'top ten': l1_table.head(10), 'the report numbers': l1_table.loc[list(names.values())],
           'near copies of them': l1_table.loc[copies]}).round(2)

penalty C = 3; numbers picked in at least half the redraws: 22


picked in this share of redraws  \
top ten             decay_residual                                   1.00   
                    train_acc                                        1.00   
                    grad_norm_W1                                     0.98   
                    grad_weight_cos                                  0.98   
                    eig_gap_top88                                    0.97   
                    fourier_dW_c_dcshare                             0.90   
                    dA_step                                          0.87   
                    fourier_dW_match                                 0.86   
                    D                                                0.82   
                    fourier_dW_a_dcshare                             0.81   
the report numbers  A_t                                              0.00   
                    S_t                                              0.01   
                    R_t                                              0.03   
near copies of them A_t_train                                        0.07   
                    A_first_logit                                    0.08   
                    S_full                                           0.04   
                    S_W1                                             0.01   
                    share_k0                                         0.03   
                    R_full                                           0.06   

                                          share of spread set by weight decay  \
top ten             decay_residual                                       0.92   
                    train_acc                                            0.74   
                    grad_norm_W1                                         0.18   
                    grad_weight_cos                                      0.59   
                    eig_gap_top88                                        0.00   
                    fourier_dW_c_dcshare                                 0.76   
                    dA_step                                              0.01   
                    fourier_dW_match                                     0.13   
                    D                                                    0.01   
                    fourier_dW_a_dcshare                                 0.68   
the report numbers  A_t                                                  0.18   
                    S_t                                                  0.56   
                    R_t                                                  0.07   
near copies of them A_t_train                                            0.16   
                    A_first_logit                                        0.20   
                    S_full                                               0.56   
                    S_W1                                                 0.49   
                    share_k0                                             0.07   
                    R_full                                               0.07   

                                          rank  
top ten             decay_residual           1  
                    train_acc                2  
                    grad_norm_W1             3  
                    grad_weight_cos          4  
                    eig_gap_top88            5  
                    fourier_dW_c_dcshare     6  
                    dA_step                  7  
                    fourier_dW_match         8  
                    D                        9  
                    fourier_dW_a_dcshare    10  
the report numbers  A_t                    106  
                    S_t                     85  
                    R_t                     79  
near copies of them A_t_train               72  
                    A_first_logit           66  
                    S_full                  76  
                    S_W1                    88 

**Verdict: not worth a section. One sentence in the appendix at most.**

- It does score higher: 0.983 against 0.960 for settings + the 3 kernel numbers, on the same folds.
- Look at what it picks, though. Three of the top four are training accuracy at step 2,000, `decay_residual` and `grad_weight_cos`. Weight decay alone sets 59 to 92 percent of their spread. So the L1 model mostly rebuilds the settings from inside the network, plus how far memorisation has got.
- It drops `A_t`, `S_t` and `R_t` in almost every redraw, and their near copies too (picked in 8 percent of redraws or fewer). Once weight decay and training accuracy are in, it finds the kernel numbers have little left to add for "whether".
- Only about 22 numbers survive in at least half of the redraws. Below the top five, the picks come and go.

The report asks about A, S and R. The L1 model answers a different question: what is the best early alarm for grokking? A proper version of that needs its own section, its own survival model and its own checks, so it's out of scope. If you want one line for the appendix: "An L1 logistic model over all 108 numbers logged at step 2,000 reaches a held-out AUC of 0.98. Its top picks track weight decay and training accuracy, and it rarely selects A_t, S_t or R_t."

## What to put in the paper

1. **Formulas.** Write the AFT and the logistic model as in Section 1. Call the AFT what it is: a regression on log T that keeps the censored runs.
2. **Main result.** Lead with the within-cell Cox model. Among the seeds of one cell, higher `A_t` at step 2,000 goes with earlier grokking, and that effect doesn't change with weight decay. This result is free of the overlap problem and the weight decay problem.
3. **Prediction.** On held-out cells, adding the kernel at step 2,000 to the settings raises the pooled concordance from 0.877 to 0.901, and the pooled AUC from 0.950 to 0.965. Report the joint likelihood ratio test for the three numbers together.
4. **Weight decay.** Give each level its own term. Say that it shifts the grokking time and changes its spread, and that the across-cell link between the kernel and the time differs by level. Don't claim the model predicts an untested level.
5. **Things to drop.** The Part 8 claim that `R_t` matters most. Any reading of the sign of `S_t` on its own. The no weight decay set as a headline. A big L1 section.
6. **Limits.** The AFT intervals treat seeds as independent, so they're too narrow. Weight decay 3e-5 was chosen after looking at the outcomes. The results use one landmark, step 2,000.